# Thermal Balance Corrector Test

## What is tested

`ThermalBalanceCorrector` rescales a relaxation superoperator so that its population
transfer rates obey thermal detailed balance. This notebook checks that the corrected
superoperator describes physically valid dynamics.

## List of tests

1. `test_detailed_balance`: population rates satisfy $w_{j \to i} / w_{i \to j} = \exp((E_j - E_i) / k_B T)$.
2. `test_trace_preservation`: every population column of the superoperator sums to zero, so $d\,\mathrm{Tr}[\rho]/dt = 0$.
3. `test_dephasing_correction`: the diagonal of the superoperator changes by $-\tfrac{1}{2}(\Delta\gamma_i + \Delta\gamma_j)$, where $\Delta\gamma$ is the change of the population out-rates. A faster out-rate makes the coherence decay faster.

## How to run

Run all cells. The last cell calls `run_tests()`. Each test prints one line when it
passes and raises `AssertionError` when it fails.

In [1]:
import os
import sys

import torch

# Makes MaRs importable when the notebook is run from the repository without installation
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import mars
from mars.population.thermal_corrections import ThermalBalanceCorrector

## Helper functions

In [2]:
def kelvin_to_hz(energies_K: list[float]) -> torch.Tensor:
    """Convert level energies from K to Hz, the units expected by ThermalBalanceCorrector."""
    return mars.constants.unit_converter(torch.tensor(energies_K, dtype=torch.float64), "K_to_Hz").to(torch.float32)


def random_symmetric_rates(dim: int, max_rate: float, seed: int) -> torch.Tensor:
    """Return a random symmetric rate matrix with a zero diagonal."""
    torch.manual_seed(seed)
    rates = torch.rand(dim, dim) * max_rate
    rates = 0.5 * (rates + rates.T)
    rates.fill_diagonal_(0.0)
    return rates


def population_indices(dim: int) -> torch.Tensor:
    """Return the positions of the populations rho_ii in the vectorized density matrix."""
    return torch.arange(dim) * (dim + 1)


def build_superoperator(rates: torch.Tensor) -> torch.Tensor:
    """Place a rate matrix into the population block of an empty N^2 x N^2 superoperator."""
    dim = rates.shape[-1]
    superop = torch.zeros(dim * dim, dim * dim)
    pop_idx = population_indices(dim)
    superop[pop_idx[:, None], pop_idx[None, :]] = rates
    return superop

## Tests

In [3]:
def test_detailed_balance():
    """Check that all population rates satisfy the Boltzmann ratio after the correction."""
    dim = 4
    energies = kelvin_to_hz([0.0, 1.5, 3.0, 5.0])
    temperature = torch.tensor(2.0)
    corrector = ThermalBalanceCorrector(energies, thermal_balance_mode="symmetric")

    mean_rates = random_symmetric_rates(dim, max_rate=0.5, seed=42)
    superop = build_superoperator(mean_rates)
    pop_idx = population_indices(dim)
    superop[pop_idx, pop_idx] = -mean_rates.sum(dim=-2)

    corrected = corrector.apply_superoperator_transform(temperature, superop)
    new_rates = corrected[pop_idx[:, None], pop_idx[None, :]]

    ratio = new_rates / new_rates.T
    expected_ratio = torch.exp(corrector.omega_K / temperature)

    # Skip the diagonal and numerically zero rates, where the ratio is undefined
    mask = ~torch.eye(dim, dtype=torch.bool) & (new_rates > 1e-8)

    assert torch.allclose(ratio[mask], expected_ratio[mask], rtol=1e-4), (
        f"Detailed balance violated. Max error: {(ratio[mask] - expected_ratio[mask]).abs().max()}"
    )
    print("Detailed balance test passed (rate ratios match the Boltzmann factor)")


def test_trace_preservation():
    """Check that each population column of the corrected superoperator sums to zero."""
    dim = 3
    energies = kelvin_to_hz([0.0, 2.0, 4.5])
    temperature = torch.tensor(1.2)
    corrector = ThermalBalanceCorrector(energies, thermal_balance_mode="symmetric")

    rates = random_symmetric_rates(dim, max_rate=0.4, seed=99)
    superop = build_superoperator(rates)
    pop_idx = population_indices(dim)
    superop[pop_idx, pop_idx] = -rates.sum(dim=-2)

    corrected = corrector.apply_superoperator_transform(temperature, superop)
    column_sums = corrected[:, pop_idx].sum(dim=0)

    assert torch.allclose(column_sums, torch.zeros_like(column_sums), atol=1e-6), (
        f"Trace is not preserved. Column sums: {column_sums}"
    )
    print("Trace preservation test passed (population columns sum to zero)")


def test_dephasing_correction():
    """Check that the superoperator diagonal changes by minus half the sum of the out-rate changes."""
    dim = 3
    energies = kelvin_to_hz([0.0, 1.0, 2.5])
    temperature = torch.tensor(1.5)
    corrector = ThermalBalanceCorrector(energies, thermal_balance_mode="symmetric")

    # The superoperator diagonal stays zero to isolate the correction term
    old_rates = random_symmetric_rates(dim, max_rate=0.25, seed=7)
    superop = build_superoperator(old_rates)
    pop_idx = population_indices(dim)

    corrected = corrector.apply_superoperator_transform(temperature, superop)

    # Out-rates are sums of the off-diagonal rates, because the new diagonal also contains the correction
    new_rates = corrected[pop_idx[:, None], pop_idx[None, :]]
    new_rates = new_rates - torch.diag(new_rates.diag())
    out_rates_change = new_rates.sum(dim=-2) - old_rates.sum(dim=-2)

    # The diagonal of the superoperator holds decay rates with a negative sign
    expected_change = -0.5 * (out_rates_change.unsqueeze(-1) + out_rates_change.unsqueeze(-2)).flatten()
    actual_change = corrected.diag() - superop.diag()

    # Population diagonals are covered by test_trace_preservation
    coherence_mask = torch.ones(dim * dim, dtype=torch.bool)
    coherence_mask[pop_idx] = False

    assert torch.allclose(
        actual_change[coherence_mask], expected_change[coherence_mask], rtol=1e-5, atol=1e-8
    ), f"Dephasing correction mismatch. Max error: {(actual_change - expected_change).abs().max()}"
    print("Dephasing correction test passed (the diagonal changes by -1/2 (dg_i + dg_j))")

## Run all tests

In [4]:
def run_tests():
    """Run every test of this notebook."""
    tests = [test_detailed_balance, test_trace_preservation, test_dephasing_correction]
    for test in tests:
        test()
    print(f"\nAll {len(tests)} thermal balance corrector tests passed.")


run_tests()

Detailed balance test passed (rate ratios match the Boltzmann factor)
Trace preservation test passed (population columns sum to zero)
Dephasing correction test passed (the diagonal changes by -1/2 (dg_i + dg_j))

All 3 thermal balance corrector tests passed.
